# Fase 4 v3 — Extremos com amostragem estratificada global

Validação da varredura global, pesos, prevalência exata e efeitos evento × não-evento.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

OUT = Path('../analysis_outputs/04_extremes')
summary = json.loads((OUT/'analysis_summary.json').read_text())
summary


## 1. Validação do scan global

In [ ]:
for key in [
    'phase_version',
    'global_patches_scanned',
    'global_valid_pixels_scanned',
    'exact_global_positive_pixel_rate',
    'retained_global_stratified_sample_rows',
    'pass_b_unique_input_blocks_read',
    'pass_b_total_input_blocks',
    'pass_b_input_block_fraction_read',
    'max_abs_fixed_threshold_rate_diff_v3_vs_phase0',
]:
    print(f'{key}: {summary.get(key)}')


## 2. Estratos globais e pesos

In [ ]:
strata = pd.read_parquet(OUT/'stratum_sampling.parquet')
display(strata)


## 3. Prevalência exata v3 × Fase 0

In [ ]:
prev = pd.read_parquet(OUT/'event_prevalence.parquet')
fixed = prev[prev.family.eq('fixed')].copy()
display(fixed[[
    'event_id','threshold','exact_global_event_count_v3',
    'exact_global_event_rate_v3','reference_event_rate_phase0',
    'abs_rate_diff_v3_vs_phase0','sample_event_count_unweighted',
    'effective_sample_size_event'
]])


In [ ]:
plot = fixed.dropna(subset=['reference_event_rate_phase0'])
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(plot.threshold, plot.exact_global_event_rate_v3, marker='o', label='Fase 4 v3 — scan global')
ax.plot(plot.threshold, plot.reference_event_rate_phase0, marker='o', label='Fase 0 — scan exato')
ax.set_yscale('log')
ax.set_xlabel('Limiar da legenda do radar')
ax.set_ylabel('Fração de pixels na distribuição em patches')
ax.legend()
ax.set_title('Validação: Fase 4 v3 × Fase 0')
plt.tight_layout()
plt.show()


## 4. Eventos e tamanho efetivo da amostra

In [ ]:
display(fixed[[
    'event_id','sample_event_count_unweighted',
    'effective_sample_size_event','exact_global_event_rate_v3'
]])


## 5. Principais efeitos evento × não-evento

In [ ]:
effects = pd.read_parquet(OUT/'effect_sizes.parquet')
for event_id in ['fixed_ge_20','fixed_ge_30','fixed_ge_40','fixed_ge_45']:
    e = effects[effects.event_id.eq(event_id)].copy()
    e['abs_smd'] = e.standardized_mean_difference.abs()
    print('\n', event_id)
    display(e.sort_values('abs_smd', ascending=False).head(10)[[
        'predictor','source','event_mean','non_event_mean',
        'standardized_mean_difference','n_event_sample','effective_n_event'
    ]])


## 6. Interpretação

`effective_n_event` é o ESS associado aos pesos globais. Ele **não** equivale ao número de eventos meteorológicos independentes, pois o dataset mantém overlap espacial e dependência temporal.